[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C4/notebooks/registrace.ipynb)

# Registrace MRI: tři parametry, jeden `backward()`

Dva snímky téhož mozku se liší tím, že pacient mezi sekvencemi pohnul hlavou.
Hledáme posun a rotaci $\mathbf p = (t_x, t_y, \theta)$, které referenční snímek
přenesou na pohnutý:

$$
\text{minimize}_{\mathbf p}\ \ \frac{1}{m}\sum_{i=1}^{m}
\bigl(T_{\mathbf p}(\mathbf a)_i - b_i\bigr)^2
$$

$T_{\mathbf p}$ = otočit a posunout obraz s bilineární interpolací, MSE se
počítá přes všech $m = 128 \cdot 128$ pixelů. Pohnutý snímek si vyrobíme sami ze
známé pravdy, takže výsledek půjde zkontrolovat. Doplňte **dvě místa `# TODO`**.

| ve vzorci | v kódu |
|---|---|
| $\mathbf a$, $\mathbf b$ | `mri`, `pohnuty` |
| $T_{\mathbf p}(\mathbf a)$ | `transformuj(mri, p)` |
| $\mathbf p$ | `p` s `requires_grad=True` |

---

*Čísla, se kterými se notebook porovnává, počítají skripty [`kod/cisla.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C4/kod/cisla.py), [`kod/registrace.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C4/kod/registrace.py).*

In [ ]:
import os, urllib.request
RAW = "https://github.com/tomasvicar/OMM-public/raw/master/cviceni/C4/data"
if not os.path.exists("mri-128.png"):
    urllib.request.urlretrieve(f"{RAW}/mri-128.png", "mri-128.png")

In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

mri = torch.from_numpy(np.asarray(Image.open("mri-128.png"), dtype=np.float32) / 255)[None, None]

def transformuj(obr, p):
    """Otočí obraz o p[2] (radiány) a posune o (p[0], p[1]) v souřadnicích -1..1."""
    c, s = torch.cos(p[2]), torch.sin(p[2])
    theta = torch.stack([torch.stack([c, -s, p[0]]),
                         torch.stack([s, c, p[1]])])[None]
    mrizka = F.affine_grid(theta, obr.shape, align_corners=False)
    return F.grid_sample(obr, mrizka, mode="bilinear", align_corners=False)

def prekryti(cil, obr):
    """cíl zeleně, obr purpurově; kde sedí, je šedá"""
    a, b = cil[0, 0].clamp(0, 1), obr[0, 0].clamp(0, 1)
    return torch.stack([b, a, b], dim=-1).numpy()

torch.manual_seed(0)
pravda = torch.tensor([0.10, -0.06, math.radians(12)])     # 6,4 px, -3,84 px, 12°
pohnuty = (transformuj(mri, pravda) + 0.02 * torch.randn_like(mri)).detach()

fig, ax = plt.subplots(1, 3, figsize=(11, 4))
ax[0].imshow(mri[0, 0], cmap="gray"); ax[0].set_title("referenční")
ax[1].imshow(pohnuty[0, 0], cmap="gray"); ax[1].set_title("pohnutý")
ax[2].imshow(prekryti(pohnuty, mri)); ax[2].set_title("překrytí před registrací")
plt.show()

## Krok 1 — optimalizační smyčka *(díra 1)*

Stejná smyčka jako při učení sítě, jen „vahami“ jsou tři čísla. Start je
$\mathbf p = (0, 0, 0)$, tedy „pacient se nepohnul“.

In [ ]:
def registruj(cil, start, kroku=300, lr=0.02):
    """Najde p = (tx, ty, theta), aby transformuj(mri, p) bylo co nejblíž cil."""
    p = torch.tensor(start, requires_grad=True)
    opt = torch.optim.Adam([p], lr=lr)
    for k in range(kroku):
        # TODO 1: jeden krok Adamu jako při učení sítě
        #   vynulovat gradient, ztrata = MSE mezi transformuj(mri, p) a cil,
        #   backward, krok optimalizátoru
        ...
    return p.detach(), ztrata.item()

In [ ]:
p, mse = registruj(pohnuty, [0.0, 0.0, 0.0])
print(f"tx = {p[0]:.4f}, ty = {p[1]:.4f}, theta = {math.degrees(p[2]):.2f}°, MSE = {mse:.6f}")
assert torch.allclose(p[:2], pravda[:2], atol=1e-3) and abs(math.degrees(p[2]) - 12) < 0.05, \
    "nesedí s pravdou (0,10; -0,06; 12°)"
assert abs(mse - 0.000403) < 2e-5, "MSE má vyjít 0,000403, tedy zhruba sigma^2 šumu"

plt.imshow(prekryti(pohnuty, transformuj(mri, p))); plt.title("překrytí po registraci")
plt.show()

## Krok 2 — velká rotace a multistart *(díra 2)*

Teď se hlava „otočí“ o 150°. Start z nuly je od pravdy hodně daleko.

In [ ]:
pravda150 = torch.tensor([0.10, -0.06, math.radians(150)])
pohnuty150 = (transformuj(mri, pravda150) + 0.02 * torch.randn_like(mri)).detach()

p0, mse0 = registruj(pohnuty150, [0.0, 0.0, 0.0])
print(f"start 0°: theta = {math.degrees(p0[2]):.2f}°, MSE = {mse0:.6f}")

In [ ]:
# TODO 2: multistart. Spusťte registruj(pohnuty150, ...) ze čtyř startů
#   theta0 = 0°, 90°, 180°, 270° (posun vždy 0) a nechte si ten s nejmenší MSE.
#   Úhel do radiánů: math.radians(...)
nejlepsi_p, nejlepsi_mse = None, float("inf")
...
print(f"nejlepší: theta = {math.degrees(nejlepsi_p[2]) % 360:.2f}°, MSE = {nejlepsi_mse:.6f}")

In [ ]:
assert abs(math.degrees(nejlepsi_p[2]) % 360 - 150) < 0.5 and nejlepsi_mse < 0.001, \
    "multistart měl najít 150°"
plt.imshow(prekryti(pohnuty150, transformuj(mri, nejlepsi_p))); plt.title("multistart: překrytí")
plt.show()